# T-SQL Fundamentals — Fourth Edition
## Chapter 08 — Data Modification | Exercises

| Field | Value |
|-------|-------|
| **Author** | Ralph Cange |
| **Group** | EOS_grp_3 |
| **Course** | CSCI331 — Database Class |
| **Date** | 04/09/2026 |
| **Database** | NorthWinds2024Student |
| **Reference** | © Itzik Ben-Gan |

---
> **Note:** The original textbook uses TSQLV6. All exercises below have been adapted to use the **NorthWinds2024Student** named database.


---
## Connection Setup
Install the `pyodbc` library and establish a connection to SQL Server before running any cells below.


In [ ]:
%pip install pyodbc sqlalchemy

import pyodbc

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost,14001;"
    "DATABASE=NorthWinds2024Student;"
    "UID=sa;"
    "PWD=PH@123456789;"
    "TrustServerCertificate=yes;"
)
cursor = conn.cursor()
conn.autocommit = True
print("Connected to NorthWinds2024Student successfully.")

---
## Exercise 1 — Table Setup: `dbo.Customers2`

Run the following code to create the `dbo.Customers2` table in the NorthWinds2024Student database.


In [ ]:
cursor.execute("""
USE NorthWinds2024Student;
""")

cursor.execute("""
DROP TABLE IF EXISTS dbo.Customers2;
""")

cursor.execute("""
CREATE TABLE dbo.Customers2
(
  custid      INT          NOT NULL PRIMARY KEY,
  companyname NVARCHAR(40) NOT NULL,
  country     NVARCHAR(15) NOT NULL,
  region      NVARCHAR(15) NULL,
  city        NVARCHAR(15) NOT NULL
);
""")
print("dbo.Customers2 created successfully.")

### Exercise 1-1 — INSERT a Single Row

Insert into `dbo.Customers2` a row with:
- **custid:** 100
- **companyname:** Coho Winery
- **country:** USA
- **region:** WA
- **city:** Redmond


In [ ]:
cursor.execute("""
INSERT INTO dbo.Customers2 (custid, companyname, country, region, city)
VALUES (100, 'Coho Winery', 'USA', 'WA', 'Redmond');
""")
print("Row inserted.")

cursor.execute("SELECT * FROM dbo.Customers2;")
rows = cursor.fetchall()
for row in rows:
    print(row)

### Exercise 1-2 — INSERT Customers Who Placed Orders

Insert into `dbo.Customers2` all customers from `Sales.Customers` who placed at least one order.

> **Fix applied:** Added `DISTINCT` to prevent PRIMARY KEY violation from customers with multiple orders.


In [ ]:
cursor.execute("""
INSERT INTO dbo.Customers2 (custid, companyname, country, region, city)
SELECT DISTINCT
    CustomerId,
    CustomerCompanyName,
    CustomerCountry,
    CustomerRegion,
    CustomerCity
FROM Sales.Customers AS C
WHERE EXISTS (
    SELECT *
    FROM Sales.[Order] AS O
    WHERE O.CustomerId = C.CustomerId
);
""")
print(f"Rows inserted: {cursor.rowcount}")

### Exercise 1-3 — SELECT INTO `dbo.Orders2` (2020–2022)

Use a `SELECT INTO` statement to create and populate `dbo.Orders2` with orders placed in the years 2020 through 2022.


In [ ]:
cursor.execute("DROP TABLE IF EXISTS dbo.Orders2;")

cursor.execute("""
SELECT *
INTO dbo.Orders2
FROM Sales.[Order]
WHERE YEAR(OrderDate) = 2020
   OR YEAR(OrderDate) = 2021
   OR YEAR(OrderDate) = 2022;
""")
print(f"Rows inserted into dbo.Orders2: {cursor.rowcount}")

---
## Exercise 2 — DELETE with OUTPUT Clause

Delete from `dbo.Orders2` all orders placed **before August 2020**.  
Use the `OUTPUT` clause to return the `orderid` and `orderdate` of the deleted rows.

**Expected:** 22 rows affected


In [ ]:
cursor.execute("""
DELETE FROM dbo.Orders2
OUTPUT
    deleted.OrderId,
    deleted.OrderDate
WHERE OrderDate < '20200801';
""")

print(f"{'orderid':<12} {'orderdate'}")
print("-" * 25)
rows = cursor.fetchall()
for row in rows:
    print(f"{row[0]:<12} {row[1]}")
print(f"\n({len(rows)} rows affected)")

---
## Exercise 3 — DELETE Orders from Brazil Customers

Delete from `dbo.Orders2` all orders placed by customers whose country is **Brazil**.

Uses a CTE with `EXISTS` to identify matching orders.

> **Note:** `Sales.Customers` stores country as `'Brazil'` (not `'BRAZIL'`). Case matters in some collations.


In [ ]:
cursor.execute("""
WITH O AS (
    SELECT *
    FROM dbo.Orders2 AS O2
    WHERE EXISTS (
        SELECT *
        FROM Sales.Customers AS C
        WHERE C.CustomerId = O2.CustomerId
          AND C.CustomerCountry = N'Brazil'
    )
)
DELETE O;
""")
print(f"Brazil customer orders deleted: {cursor.rowcount} rows affected")

---
## Exercise 4 — UPDATE with OUTPUT Clause

Update `dbo.Customers2` and change all `NULL` region values to `'<None>'`.  
Use the `OUTPUT` clause to show `custid`, old region, and new region.

**Expected:** 58 rows affected


In [ ]:
cursor.execute("""
UPDATE dbo.Customers2
SET region = '<None>'
OUTPUT
    inserted.custid,
    deleted.region  AS oldregion,
    inserted.region AS newregion
WHERE region IS NULL;
""")

print(f"{'custid':<12} {'oldregion':<16} {'newregion'}")
print("-" * 40)
rows = cursor.fetchall()
for row in rows:
    print(f"{row[0]:<12} {str(row[1]):<16} {row[2]}")
print(f"\n({len(rows)} rows affected)")

---
## Exercise 5 — MERGE: Update UK Orders with Customer Info

Update all orders in `dbo.Orders2` placed by **UK customers**.  
Set `ShipToCountry`, `ShipToRegion`, `ShipToCity` to match  
the corresponding customer values from `dbo.Customers2`.


In [ ]:
cursor.execute("""
MERGE INTO dbo.Orders2 AS O
USING (
    SELECT *
    FROM dbo.Customers2
    WHERE country = N'UK'
) AS C
ON O.CustomerId = C.custid
WHEN MATCHED THEN
    UPDATE SET
        ShipToCountry = C.country,
        ShipToRegion  = C.region,
        ShipToCity    = C.city;
""")
print(f"UK orders updated: {cursor.rowcount} rows affected")

---
## Exercise 6 — Orders2 + OrderDetails Setup

Recreate `dbo.Orders2` and `dbo.OrderDetails` with full schema,  
constraints, and foreign keys. Then populate from `Sales.[Order]`  
and `Sales.OrderDetail`.


In [ ]:
cursor.execute("DROP TABLE IF EXISTS dbo.OrderDetails;")
cursor.execute("DROP TABLE IF EXISTS dbo.Orders2;")

cursor.execute("""
CREATE TABLE dbo.Orders2
(
  orderid        INT          NOT NULL,
  custid         INT          NULL,
  empid          INT          NOT NULL,
  shipperid      INT          NOT NULL,
  orderdate      DATE         NOT NULL,
  requireddate   DATE         NOT NULL,
  shippeddate    DATE         NULL,
  freight        MONEY        NOT NULL
    CONSTRAINT DFT_Orders_freight DEFAULT(0),
  shipname       NVARCHAR(40) NOT NULL,
  shipaddress    NVARCHAR(60) NOT NULL,
  shipcity       NVARCHAR(15) NOT NULL,
  shipregion     NVARCHAR(15) NULL,
  shippostalcode NVARCHAR(10) NULL,
  shipcountry    NVARCHAR(15) NOT NULL,
  CONSTRAINT PK_Orders PRIMARY KEY(orderid)
);
""")

cursor.execute("""
CREATE TABLE dbo.OrderDetails
(
  orderid   INT           NOT NULL,
  productid INT           NOT NULL,
  unitprice MONEY         NOT NULL
    CONSTRAINT DFT_OrderDetails_unitprice DEFAULT(0),
  qty       SMALLINT      NOT NULL
    CONSTRAINT DFT_OrderDetails_qty DEFAULT(1),
  discount  NUMERIC(4, 3) NOT NULL
    CONSTRAINT DFT_OrderDetails_discount DEFAULT(0),
  CONSTRAINT PK_OrderDetails PRIMARY KEY(orderid, productid),
  CONSTRAINT FK_OrderDetails_Orders FOREIGN KEY(orderid)
    REFERENCES dbo.Orders2(orderid),
  CONSTRAINT CHK_discount  CHECK (discount BETWEEN 0 AND 1),
  CONSTRAINT CHK_qty       CHECK (qty > 0),
  CONSTRAINT CHK_unitprice CHECK (unitprice >= 0)
);
""")
print("Tables created successfully.")

In [ ]:
cursor.execute("""
INSERT INTO dbo.Orders2
SELECT * FROM Sales.[Order];
""")
print(f"Orders2 populated: {cursor.rowcount} rows")

cursor.execute("""
INSERT INTO dbo.OrderDetails (orderid, productid, unitprice, qty, discount)
SELECT OrderId, ProductId, UnitPrice, Quantity, DiscountPercentage
FROM Sales.OrderDetail;
""")
print(f"OrderDetails populated: {cursor.rowcount} rows")

### Exercise 6 — TRUNCATE Both Tables

Truncate `dbo.OrderDetails` and `dbo.Orders2`.

> **Fix applied:** Must DROP the foreign key constraint first before truncating,  
> then TRUNCATE the child table before the parent table.


In [ ]:
# Step 1 — Drop FK constraint so TRUNCATE is allowed
cursor.execute("""
ALTER TABLE dbo.OrderDetails
DROP CONSTRAINT FK_OrderDetails_Orders;
""")
print("Foreign key constraint dropped.")

# Step 2 — Truncate child table first
cursor.execute("TRUNCATE TABLE dbo.OrderDetails;")
print("dbo.OrderDetails truncated.")

# Step 3 — Truncate parent table
cursor.execute("TRUNCATE TABLE dbo.Orders2;")
print("dbo.Orders2 truncated.")

---
## Cleanup

Drop all tables created during this exercise session.


In [ ]:
cursor.execute("DROP TABLE IF EXISTS dbo.OrderDetails;")
cursor.execute("DROP TABLE IF EXISTS dbo.Orders2;")
cursor.execute("DROP TABLE IF EXISTS dbo.Customers2;")
print("All tables dropped. Cleanup complete.")
conn.close()
print("Connection closed.")